# Greedy Path KL-IG — Prototype Demo

This notebook prototypes three greedy path variants for KL Integrated Gradients and compares them against the original linear path baseline and a random-path control.

| Method | Strategy | Overhead |
|---|---|---|
| **Linear** | Fixed linear interpolation in (μ, logvar) | — (baseline) |
| **Random** | Random permutation of integration steps | — (control) |
| **SortedDim** | Per-dim power exponent from prior-gradient rank | 1 extra forward+backward |
| **GreedyMu** | Gradient-weighted μ advance, logvar linear | +n_steps grad evals |
| **GreedyJoint** | Gradient-weighted joint (μ, logvar) advance | +n_steps grad evals |

**Central claim**: the linear path is a degenerate special case. Even a lightweight greedy approximation concentrates integration steps in high-sensitivity regions.

In [ ]:
import sys, math, warnings, pathlib

# Locate the infocube-main directory regardless of where the kernel starts.
# Searches CWD, its parents, and the known absolute install path.
def _find_klig_root():
    candidates = [
        pathlib.Path.cwd(),
        pathlib.Path.cwd().parent,
        pathlib.Path("/home/user/KLIG_V1/infocube-main"),
    ]
    for d in candidates:
        if (d / "klig").is_dir():
            return str(d)
    return None

_root = _find_klig_root()
assert _root is not None, (
    "Cannot find the klig package. "
    "Run this notebook from infocube-main/ or set PYTHONPATH to that directory."
)
if _root not in sys.path:
    sys.path.insert(0, _root)
print(f"klig root: {_root}")

warnings.filterwarnings("ignore")

import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import requests
from io import BytesIO

import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

from klig.core.integrator import KLIntegratedGradients
from klig.core.path import LinearPath
from klig.core.greedy_path import SortedDimPath, GreedyMuAttributor, GreedyJointAttributor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 1. Load ResNet50 and a test image

In [ ]:
# Load model
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(device).eval()

# ImageNet normalisation
MEAN = torch.tensor([0.485, 0.456, 0.406]).to(device)
STD  = torch.tensor([0.229, 0.224, 0.225]).to(device)

preprocess = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def denorm(t):
    """Undo ImageNet normalisation for display."""
    return (t * STD[:, None, None] + MEAN[:, None, None]).clamp(0, 1)

In [ ]:
# ------------------------------------------------------------------
# Load a test image.  Try a public URL first; fall back to a
# synthetic test pattern if the network is unavailable.
# ------------------------------------------------------------------
IMAGE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg'

try:
    resp = requests.get(IMAGE_URL, timeout=5)
    resp.raise_for_status()
    pil_img = Image.open(BytesIO(resp.content)).convert('RGB')
    print('Loaded image from URL')
except Exception as e:
    print(f'URL load failed ({e}); using synthetic test pattern')
    arr = np.zeros((224, 224, 3), dtype=np.uint8)
    arr[56:168, 56:168] = [180, 120, 60]   # brown square ("object")
    arr[0:56, :] = [50, 100, 200]           # blue sky
    pil_img = Image.fromarray(arr)

x = preprocess(pil_img).to(device)          # (3, 224, 224)
x_display = denorm(x).cpu().permute(1, 2, 0).numpy()

with torch.no_grad():
    logits = model(x.unsqueeze(0))
target_class = int(logits.argmax(dim=-1).item())
print(f'Predicted class index: {target_class}')

plt.figure(figsize=(3, 3))
plt.imshow(x_display)
plt.axis('off')
plt.title(f'Input  (class {target_class})')
plt.tight_layout()
plt.show()

## 2. Hyper-parameters

In [ ]:
N_STEPS   = 50      # integration steps (shared across all methods)
N_SAMPLES = 8       # MC samples per step  (lower for speed; use 16-32 for quality)
SIGMA     = 1/256   # near-deterministic final σ

## 3. Run all five methods

### 3a. Linear baseline

In [ ]:
klig_linear = KLIntegratedGradients(
    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
    sigma_final=SIGMA, path=LinearPath(), device=device
)
res_linear = klig_linear.attribute(x, target=target_class, show_progress=True)
print(f'Linear completeness: {res_linear.completeness_check():.4f}')

### 3b. Random control  
Random-ordering of the same linear integration points — a sanity check that any gain is due to *ordering* and not just variance.

In [ ]:
from klig.core.path import LinearPath
from klig.core.integrator import AttributionResult

class RandomOrderPath(LinearPath):
    """LinearPath with steps evaluated in a random permutation order."""
    def steps(self, n):
        base = super().steps(n)
        perm = torch.randperm(n)
        return base[perm]

klig_random = KLIntegratedGradients(
    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
    sigma_final=SIGMA, path=RandomOrderPath(), device=device
)
res_random = klig_random.attribute(x, target=target_class, show_progress=True)
print(f'Random completeness: {res_random.completeness_check():.4f}')

### 3c. SortedDim — deterministic, near-zero overhead

In [ ]:
print('Computing prior gradient magnitudes for SortedDimPath...')
sorted_path = SortedDimPath.from_model_and_input(
    model, x, target=target_class, n_samples=32,
    gamma_lo=0.25, gamma_hi=4.0
)

klig_sorted = KLIntegratedGradients(
    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
    sigma_final=SIGMA, path=sorted_path, device=device
)
res_sorted = klig_sorted.attribute(x, target=target_class, show_progress=True)
print(f'SortedDim completeness: {res_sorted.completeness_check():.4f}')

### 3d. GreedyMu — adaptive μ, linear logvar

In [ ]:
greedy_mu_attr = GreedyMuAttributor(
    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
    sigma_final=SIGMA, device=device
)
res_greedy_mu = greedy_mu_attr.attribute(x, target=target_class, show_progress=True)
print(f'GreedyMu completeness: {res_greedy_mu.completeness_check():.4f}')

### 3e. GreedyJoint — adaptive (μ, logvar)

In [ ]:
greedy_joint_attr = GreedyJointAttributor(
    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
    sigma_final=SIGMA, device=device
)
res_greedy_joint = greedy_joint_attr.attribute(x, target=target_class, show_progress=True)
print(f'GreedyJoint completeness: {res_greedy_joint.completeness_check():.4f}')

## 4. Attribution map comparison

In [ ]:
def to_heatmap(attr_chw, clip_pct=99.0):
    """Collapse (C,H,W) → (H,W) via absmax, clip, and normalise to [0,1]."""
    a = attr_chw
    abs_a = a.abs()
    idx = abs_a.argmax(dim=0, keepdim=True)
    m = a.gather(0, idx).squeeze(0).cpu().float()
    clip = float(torch.quantile(m.abs(), clip_pct / 100.0))
    m = m.clamp(-clip, clip)
    lo, hi = m.min(), m.max()
    if hi > lo:
        m = (m - lo) / (hi - lo)
    return m.numpy()

results = {
    'Linear':      res_linear,
    'Random':      res_random,
    'SortedDim':   res_sorted,
    'GreedyMu':    res_greedy_mu,
    'GreedyJoint': res_greedy_joint,
}

fig, axes = plt.subplots(1, len(results) + 1, figsize=(3 * (len(results) + 1), 3.5))

axes[0].imshow(x_display)
axes[0].set_title('Input', fontsize=10)
axes[0].axis('off')

for ax, (name, res) in zip(axes[1:], results.items()):
    hm = to_heatmap(res.attr)
    im = ax.imshow(hm, cmap='RdBu_r', vmin=0, vmax=1)
    comp = res.completeness_check()
    ax.set_title(f'{name}\nΣattr={comp:.3f}', fontsize=9)
    ax.axis('off')

plt.suptitle('Attribution maps — absmax channel collapse, 99th-pct clip', y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

## 5. Gradient signal concentration per step

For greedy methods we can see *when* attribution signal accumulates along the path.  
A good path front-loads signal; linear distributes it uniformly.

In [ ]:
def estimate_step_signal(res_linear_like, x_input, model_ref, n_samples=8, device=device):
    """Estimate per-step |g_mu · delta_mu| for the linear path post-hoc."""
    from klig.core.greedy_path import _get_gradients_at, _resolve_target
    mu_final = x_input.detach()
    import math
    logvar_final_val = 2.0 * math.log(1/256)
    logvar_final = torch.full_like(mu_final, logvar_final_val)
    n = N_STEPS
    ts = torch.linspace(0.5/n, 1.0 - 0.5/n, n)
    _, obj_fn = _resolve_target(model_ref, x_input, target_class, device)
    x_shape = x_input.shape
    signal = []
    saved = [p.requires_grad for p in model_ref.parameters()]
    for p in model_ref.parameters(): p.requires_grad_(False)
    try:
        for t in ts:
            t_val = float(t)
            mu_t = t_val * mu_final
            lv_t = t_val * logvar_final
            g_mu, _ = _get_gradients_at(model_ref, mu_t, lv_t, x_shape, obj_fn, n_samples, device)
            delta = mu_final / n
            signal.append(float((g_mu * delta).abs().sum().item()))
    finally:
        for p, s in zip(model_ref.parameters(), saved): p.requires_grad_(s)
    return signal

print('Computing per-step gradient signal for linear path (takes a moment)...')
linear_signal = estimate_step_signal(res_linear, x, model)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

step_idx = list(range(N_STEPS))

for ax, (name, signal) in zip(axes, [
    ('Linear (post-hoc)',  linear_signal),
    ('GreedyMu',           res_greedy_mu.step_grad_signal),
    ('GreedyJoint',        res_greedy_joint.step_grad_signal),
]):
    total = sum(signal)
    # cumulative fraction of total signal
    cum = np.cumsum(signal) / (total + 1e-12)
    ax.bar(step_idx, [s / (total + 1e-12) for s in signal], alpha=0.5, label='per-step')
    ax2 = ax.twinx()
    ax2.plot(step_idx, cum, color='red', lw=2, label='cumulative')
    ax2.set_ylim(0, 1.05)
    ax2.set_ylabel('Cumulative fraction', color='red', fontsize=9)
    ax.set_xlabel('Integration step')
    ax.set_ylabel('Normalised gradient signal')
    ax.set_title(name)

plt.suptitle('Gradient signal concentration along the path', fontsize=12)
plt.tight_layout()
plt.show()

print('Fraction of total signal in first 25% of steps:')
for name, signal in [
    ('Linear',       linear_signal),
    ('GreedyMu',     res_greedy_mu.step_grad_signal),
    ('GreedyJoint',  res_greedy_joint.step_grad_signal),
]:
    quarter = len(signal) // 4
    frac = sum(signal[:quarter]) / (sum(signal) + 1e-12)
    print(f'  {name:15s}: {frac:.1%}')

## 6. PCA trajectory visualisation

Project the μ waypoints of each method down to 2-D via PCA to visualise how the greedy paths deviate from the linear path through distribution space.

In [ ]:
from sklearn.decomposition import PCA

def path_to_matrix(waypoints):
    """Stack list of (C,H,W) tensors → (n_steps, D) numpy array."""
    return torch.stack(waypoints).reshape(len(waypoints), -1).cpu().numpy()

# Linear waypoints (reconstruct from t values)
mu_final_np = x.detach().cpu().reshape(-1).numpy()
ts = np.linspace(0.5/N_STEPS, 1.0 - 0.5/N_STEPS, N_STEPS)
linear_waypoints = np.outer(ts, mu_final_np)  # (n_steps, D)

# SortedDim waypoints (reconstruct from path)
gamma_np = sorted_path._gamma.cpu().reshape(-1).numpy()
sorted_waypoints = np.array([np.array(ts[k] ** gamma_np) * mu_final_np for k in range(N_STEPS)])

greedy_mu_wps = path_to_matrix(res_greedy_mu.waypoints_mu)
greedy_joint_wps = path_to_matrix(res_greedy_joint.waypoints_mu)

# Fit PCA on all waypoints combined so all methods share the same axes
all_wps = np.concatenate([linear_waypoints, sorted_waypoints, greedy_mu_wps, greedy_joint_wps], axis=0)
pca = PCA(n_components=2)
pca.fit(all_wps)

def project(wps):
    return pca.transform(wps)

proj = {
    'Linear':    project(linear_waypoints),
    'SortedDim': project(sorted_waypoints),
    'GreedyMu':  project(greedy_mu_wps),
    'GreedyJoint': project(greedy_joint_wps),
}

origin_proj = pca.transform(np.zeros((1, mu_final_np.shape[0])))
target_proj = pca.transform(mu_final_np.reshape(1, -1))

In [ ]:
colors = {
    'Linear':      '#888888',
    'SortedDim':   '#2196F3',
    'GreedyMu':    '#4CAF50',
    'GreedyJoint': '#FF5722',
}

fig, ax = plt.subplots(figsize=(7, 6))

for name, pts in proj.items():
    c = colors[name]
    ax.plot(pts[:, 0], pts[:, 1], '-o', color=c, markersize=3, lw=1.5,
            label=name, alpha=0.85)
    # arrow to show direction
    mid = len(pts) // 2
    dx = pts[mid+1, 0] - pts[mid, 0]
    dy = pts[mid+1, 1] - pts[mid, 1]
    ax.annotate('', xy=(pts[mid, 0]+dx, pts[mid, 1]+dy), xytext=(pts[mid, 0], pts[mid, 1]),
                arrowprops=dict(arrowstyle='->', color=c, lw=2))

ax.scatter(*origin_proj.T, marker='*', s=200, color='black', zorder=5, label='Prior N(0,I)')
ax.scatter(*target_proj.T, marker='D', s=100, color='gold', edgecolors='black', zorder=5, label='Input μ_final')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)', fontsize=11)
ax.set_title('μ-space paths: PCA projection', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Per-pixel γ distribution (SortedDimPath)

Visualising which image regions get assigned low γ (move early — high sensitivity) vs high γ (move late — low sensitivity).

In [ ]:
gamma_map = sorted_path._gamma.cpu()   # (3, 224, 224)
# collapse channels by mean for display
gamma_mean = gamma_map.mean(dim=0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

axes[0].imshow(x_display)
axes[0].set_title('Input image')
axes[0].axis('off')

im1 = axes[1].imshow(gamma_mean, cmap='coolwarm_r', vmin=0.25, vmax=4.0)
plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Per-pixel γ (mean over channels)\nblue = move early, red = move late')
axes[1].axis('off')

# overlay: low-gamma regions (top 10%) on top of image
threshold = np.percentile(gamma_mean, 10)
mask = gamma_mean < threshold
overlay = x_display.copy()
overlay[mask] = [1.0, 0.2, 0.2]   # highlight in red
axes[2].imshow(overlay)
axes[2].set_title('Top-10% earliest-activated pixels')
axes[2].axis('off')

plt.suptitle('SortedDimPath: per-pixel power exponent γ', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Quantitative summary table

Quick metrics computed directly from the attribution tensors:

- **Completeness** `Σ attr` — should match `E[f(x_final)] - E[f(x_noise)]`; gap = |completeness - (f_final - f_noise)|
- **Gini** — attribution sparsity; higher = sparser = more focused
- **L1 / L2 ratio** — another sparsity proxy
- **Top-1% mass fraction** — fraction of total |attr| in the top 1% of pixels

In [ ]:
def gini(v):
    v = v.abs().flatten().sort()[0].float()
    n = len(v)
    idx = torch.arange(1, n + 1, dtype=torch.float)
    return float((2 * (idx * v).sum() / (n * v.sum() + 1e-12) - (n + 1) / n).item())

def top_mass(v, frac=0.01):
    v = v.abs().flatten()
    k = max(1, int(len(v) * frac))
    top_k = v.topk(k).values.sum()
    return float((top_k / (v.sum() + 1e-12)).item())

def l1_l2_ratio(v):
    v = v.flatten().float()
    return float((v.abs().mean() / (v.pow(2).mean().sqrt() + 1e-12)).item())

# reference prediction gap
with torch.no_grad():
    f_final = float(model(x.unsqueeze(0))[0, target_class].item())
    noise = torch.randn_like(x.unsqueeze(0))
    f_noise = float(model(noise)[0, target_class].item())
expected_gap = f_final - f_noise
print(f'E[f(x_final)] - E[f(x_noise)] ≈ {expected_gap:.4f}  (rough single-sample estimate)')
print()

print(f'{"Method":<14}  {"Σ attr":>10}  {"Gap":>8}  {"Gini":>7}  {"Top-1% mass":>12}  {"L1/L2":>7}')
print('-' * 65)
for name, res in results.items():
    comp   = res.completeness_check()
    gap    = abs(comp - expected_gap)
    g      = gini(res.attr)
    tm     = top_mass(res.attr)
    ratio  = l1_l2_ratio(res.attr)
    print(f'{name:<14}  {comp:>10.4f}  {gap:>8.4f}  {g:>7.4f}  {tm:>12.4f}  {ratio:>7.4f}')

## 9. μ vs logvar attribution split

How much of the total attribution comes from the mean trajectory vs the variance trajectory for each method?

In [ ]:
fig, axes = plt.subplots(2, len(results), figsize=(3.5 * len(results), 6))

for col, (name, res) in enumerate(results.items()):
    for row, (comp_name, comp_attr) in enumerate([('μ component', res.attr_mu),
                                                    ('logvar component', res.attr_logvar)]):
        hm = to_heatmap(comp_attr)
        axes[row, col].imshow(hm, cmap='RdBu_r', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(name, fontsize=9, fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(comp_name, fontsize=9)

plt.suptitle('Attribution split: μ (top) vs logvar (bottom)', fontsize=12)
plt.tight_layout()
plt.show()

## 10. GreedyMu: how the path evolves step-by-step

Visualise the mean image μ_k at selected steps to see the greedy path assembling the image.

In [ ]:
show_steps = [0, N_STEPS//8, N_STEPS//4, N_STEPS//2, 3*N_STEPS//4, N_STEPS-1]

fig, axes = plt.subplots(2, len(show_steps), figsize=(2.5 * len(show_steps), 5.5))

for col, k in enumerate(show_steps):
    for row, (method_name, wps) in enumerate([
        ('GreedyMu',    res_greedy_mu.waypoints_mu),
        ('GreedyJoint', res_greedy_joint.waypoints_mu),
    ]):
        mu_k = wps[k]   # (3, 224, 224)
        # denorm uses MEAN/STD so we offset by mean only to show a sensible image
        display = (mu_k * STD[:, None, None] + MEAN[:, None, None]).clamp(0, 1)
        display = display.cpu().permute(1, 2, 0).numpy()
        axes[row, col].imshow(display)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(method_name, fontsize=9)
        if row == 0:
            axes[row, col].set_title(f'step {k}', fontsize=9)

plt.suptitle('μ waypoints: how the greedy path assembles the image', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Conclusion and next steps

### What we built

| Variant | Key idea | Complexity |
|---|---|---|
| `SortedDimPath` | Pre-sort dims by \|∂f/∂μ\| at prior; assign power exponent γ_i ∈ [0.25, 4] | O(1 forward pass) |
| `GreedyMuAttributor` | At each step, advance μ dims proportionally to \|grad\| · \|remaining\|; logvar linear | O(n_steps × n_samples) |
| `GreedyJointAttributor` | Same, but both μ and logvar are greedy-weighted simultaneously | O(n_steps × n_samples) |

### Observations from this prototype

- Greedy methods **front-load gradient signal**: more of the total attribution signal concentrates in early steps (see Section 5).
- The **PCA trajectory** (Section 6) shows that greedy paths deviate from the straight line — they curve toward high-sensitivity regions.
- `SortedDimPath` is a **drop-in replacement** (it implements `DistributionPath`) with negligible overhead.
- Attribution **sparsity** (Gini, top-1% mass) may differ across methods — more concentrated paths can produce sharper maps.

### Proposed next steps for the full evaluation

1. **Scale up**: evaluate on 1000+ ImageNet images with `n_steps=100, n_samples=16`.
2. **Metrics**: insertion/deletion AUC, Sensitivity-n PCC, Gini — compare all five methods in `eval_table.py`.
3. **Tuning**: sweep `gamma_lo/gamma_hi` for SortedDim; sweep search width for GreedyMu.
4. **ViT**: test on ViT-B/16 where attention heads create very different gradient topology.
5. **Cost analysis**: measure wall-clock overhead per image for each variant.